# Exercises Pack: MCP Client with an LLM (Beginner, STDIO)

The architecture we build:

```
 user prompt ─► LLM (stub or real) ─► tool_calls ─► MCP client ──STDIO──► MCP server (add, multiply, resource)
                     ▲                                   │
                     └──────────── tool results ◄────────┘
```

- The **MCP client** discovers the server's tools and converts them into **LLM function specs**.
- The LLM **plans** which tool calls to make. The client **executes** them over MCP and prints the results.
- The **default mode is a stub planner**: no token, deterministic, runs anywhere. Set `GITHUB_TOKEN` (a GitHub personal access token with access to *GitHub Models*) to use a real LLM (`gpt-4o-mini`) through the OpenAI-compatible GitHub Models endpoint.

## Install & config

In [1]:
# mcp 2.x renamed FastMCP -> MCPServer; this pack uses the v1 API, so pin mcp<2
%pip install -q "mcp[cli]<2" openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, re, sys, asyncio
from importlib.metadata import version

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata          # Colab secret named GITHUB_TOKEN
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass

USE_REAL_LLM = bool(GITHUB_TOKEN)                  # toggle: stub (default) or real LLM
LLM_MODEL = "openai/gpt-4o-mini"
LLM_ENDPOINT = "https://models.github.ai/inference"
print(f"mcp {version('mcp')} | mode: {'REAL LLM (' + LLM_MODEL + ')' if USE_REAL_LLM else 'STUB planner (no token needed)'}")

mcp 1.30.0 | mode: STUB planner (no token needed)


## Exercise 1: Theory: why is STDIO simpler than HTTP for local MCP development?

- **No networking at all:** the client simply **spawns the server as a child process** and exchanges JSON-RPC messages over its stdin/stdout. There is no port to choose or free up, no host/URL configuration, no firewall, CORS or TLS.
- **No authentication or session management:** only the parent process can talk to the server. With HTTP (Streamable HTTP/SSE), anything that can reach the port can call it, so you must add auth, origin checks and session IDs.
- **Lifecycle for free:** the server starts when the client connects and dies when the client exits. No separate deployment, daemon or "is the server running?" problems, and a code change is picked up on the next run.
- **Simple debugging:** one command (`python client.py`) runs everything. Server logs go to stderr, and the server can be tested alone with `mcp run server.py`.
- **Trade-offs:** STDIO is **1 client ↔ 1 local server**. It cannot be shared across machines or users, and it cannot scale horizontally. HTTP is the choice for remote, multi-tenant or hosted servers. For learning and local tools (IDEs, desktop assistants), STDIO is the simplest correct option.

## The MCP server (`server.py`): `add` tool + one resource

In [3]:
%%writefile server.py
import logging, sys
from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="[server] %(message)s")   # never log to stdout on STDIO
mcp = FastMCP("CalcDemo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers a and b and return the sum."""
    logging.info("add(%s, %s)", a, b)
    return a + b


@mcp.resource("info://about")
def about() -> str:
    """Short description of this server."""
    return "CalcDemo MCP server: exposes arithmetic tools for LLM agents."


if __name__ == "__main__":
    mcp.run()

Writing server.py


## Exercise 2: Connect & initialize
`StdioServerParameters(command="mcp", args=["run", "server.py"])` describes how to **start** the server; `stdio_client` spawns it and returns the read/write streams; `ClientSession.initialize()` performs the MCP handshake.

> In Jupyter/Colab, `sys.stderr` is not a real file, so we redirect the server's stderr (its logs) to `server.log` with `errlog=`; otherwise the subprocess cannot be spawned.

In [4]:
from contextlib import asynccontextmanager
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)

@asynccontextmanager
async def mcp_session():
    with open("server.log", "a") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                init = await session.initialize()
                yield session, init

async def ex2():
    async with mcp_session() as (session, init):
        print("✅ initialized")
        print("server:", init.serverInfo.name, init.serverInfo.version, "| protocol:", init.protocolVersion)
        print("capabilities:", {k: v is not None for k, v in init.capabilities.model_dump().items()})

await ex2()

✅ initialized
server: CalcDemo 1.30.0 | protocol: 2025-11-25
capabilities: {'experimental': True, 'logging': False, 'prompts': True, 'resources': True, 'tools': True, 'completions': False, 'tasks': False}


## Exercise 3: Discover resources and tools

In [5]:
async def discover(show=True):
    async with mcp_session() as (session, _):
        resources = (await session.list_resources()).resources
        tools = (await session.list_tools()).tools
        if show:
            print("Resources:")
            for r in resources:
                print(f"  - {r.uri}  ({r.name}: {r.description})")
            print("Tools:")
            for t in tools:
                print(f"  - {t.name}: {t.description}")
                print(f"      inputSchema properties: {t.inputSchema.get('properties')}  required: {t.inputSchema.get('required')}")
            about = await session.read_resource("info://about")
            print("info://about ->", about.contents[0].text)
        return tools

mcp_tools = await discover()

Resources:
  - info://about  (about: Short description of this server.)
Tools:
  - add: Add two integers a and b and return the sum.
      inputSchema properties: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}  required: ['a', 'b']
info://about -> CalcDemo MCP server: exposes arithmetic tools for LLM agents.


## Exercise 4: Convert MCP tools into LLM function specs
LLMs with function calling (OpenAI-style APIs, GitHub Models, Gemini…) expect each tool as:
`{"type": "function", "function": {"name", "description", "parameters": <JSON Schema>}}`.
An MCP tool already carries a JSON Schema in `inputSchema`, so the conversion is a direct mapping.

In [6]:
def convert_to_llm_tool(tool):
    schema = dict(tool.inputSchema or {"type": "object", "properties": {}})
    schema.pop("title", None)                                   # not needed by the LLM
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or f"MCP tool {tool.name}",
            "parameters": {"type": "object",
                           "properties": schema.get("properties", {}),
                           "required": schema.get("required", [])},
        },
    }

functions = [convert_to_llm_tool(t) for t in mcp_tools]
print(json.dumps(functions, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "add",
      "description": "Add two integers a and b and return the sum.",
      "parameters": {
        "type": "object",
        "properties": {
          "a": {
            "title": "A",
            "type": "integer"
          },
          "b": {
            "title": "B",
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ]
      }
    }
  }
]


## Exercise 5: Plan & execute
- **Stub planner:** a tiny rule-based "LLM" that reads the prompt, picks a tool **from the provided function specs** (by name or keyword), and extracts the integer arguments. It returns OpenAI-style `tool_calls`. It is deterministic and needs no token.
- **Real LLM:** the same function specs are sent to `gpt-4o-mini` on GitHub Models with `tool_choice="auto"`.
- **Executor:** for each tool call, `session.call_tool(name, args)` runs it on the MCP server, and the result is printed.

In [7]:
KEYWORDS = {"add": ["add", "plus", "sum", "+"], "multiply": ["multiply", "times", "product", "*", "x"]}

def stub_llm(prompt, functions):
    available = {f["function"]["name"]: f["function"] for f in functions}
    text = prompt.lower()
    numbers = [int(n) for n in re.findall(r"-?\d+", prompt)]
    chosen = None
    for name in available:                                       # tool whose name/keyword appears in the prompt
        if any(re.search(rf"(?<![a-z]){re.escape(k)}(?![a-z])", text) for k in KEYWORDS.get(name, [name])):
            chosen = name
            break
    if chosen is None or len(numbers) < 2:
        return {"content": "I can't map this request to an available tool.", "tool_calls": []}
    params = list(available[chosen]["parameters"]["properties"])  # e.g. ["a", "b"]
    args = dict(zip(params, numbers[:len(params)]))
    return {"content": None, "tool_calls": [{"id": "call_1", "type": "function",
                                             "function": {"name": chosen, "arguments": json.dumps(args)}}]}

def real_llm(prompt, functions):
    from openai import OpenAI
    client = OpenAI(base_url=LLM_ENDPOINT, api_key=GITHUB_TOKEN)
    resp = client.chat.completions.create(
        model=LLM_MODEL, tools=functions, tool_choice="auto",
        messages=[{"role": "system", "content": "You are a helpful assistant. Use the provided tools for arithmetic."},
                  {"role": "user", "content": prompt}])
    msg = resp.choices[0].message
    return {"content": msg.content,
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                           for tc in (msg.tool_calls or [])]}

def plan(prompt, functions):
    return (real_llm if USE_REAL_LLM else stub_llm)(prompt, functions)

async def plan_and_execute(prompt):
    async with mcp_session() as (session, _):
        tools = (await session.list_tools()).tools
        functions = [convert_to_llm_tool(t) for t in tools]          # rebuilt from the server every time
        decision = plan(prompt, functions)
        print(f"PROMPT: {prompt!r}")
        print(f"  available tools: {[f['function']['name'] for f in functions]}")
        if not decision["tool_calls"]:
            print("  LLM answer (no tool call):", decision["content"])
            return
        for tc in decision["tool_calls"]:
            name, args = tc["function"]["name"], json.loads(tc["function"]["arguments"])
            print(f"  planned tool_call: {name}({args})")
            result = await session.call_tool(name, arguments=args)
            value = result.structuredContent.get("result") if result.structuredContent else result.content[0].text
            print(f"  ✅ executed via MCP -> {value}" + ("  (tool error)" if result.isError else ""))

await plan_and_execute("Add 2 to 20.")

PROMPT: 'Add 2 to 20.'
  available tools: ['add']
  planned tool_call: add({'a': 2, 'b': 20})
  ✅ executed via MCP -> 22


In [8]:
for p in ["What is 15 plus 27?", "Multiply 6 by 7", "Tell me a joke"]:
    await plan_and_execute(p)
    print()

PROMPT: 'What is 15 plus 27?'
  available tools: ['add']
  planned tool_call: add({'a': 15, 'b': 27})
  ✅ executed via MCP -> 42



PROMPT: 'Multiply 6 by 7'
  available tools: ['add']
  LLM answer (no tool call): I can't map this request to an available tool.



PROMPT: 'Tell me a joke'
  available tools: ['add']
  LLM answer (no tool call): I can't map this request to an available tool.



With only `add` on the server, *"Multiply 6 by 7"* cannot be planned: the planner only proposes tools that the server actually exposes. The LLM's capabilities are bounded by MCP discovery.

## Optional: add `multiply(a, b)`, rebuild the function list, rerun

In [9]:
%%writefile -a server.py


@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two integers a and b and return the product."""
    logging.info("multiply(%s, %s)", a, b)
    return a * b

Appending to server.py


In [10]:
# The `if __name__ == "__main__"` block is now above the new tool; `mcp run` imports the module and
# starts the `mcp` object itself, so the tool appended at the end is still registered.
mcp_tools = await discover()
functions = [convert_to_llm_tool(t) for t in mcp_tools]
print("\nRebuilt function list:", [f["function"]["name"] for f in functions])

Resources:
  - info://about  (about: Short description of this server.)
Tools:
  - add: Add two integers a and b and return the sum.
      inputSchema properties: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}  required: ['a', 'b']
  - multiply: Multiply two integers a and b and return the product.
      inputSchema properties: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}  required: ['a', 'b']
info://about -> CalcDemo MCP server: exposes arithmetic tools for LLM agents.

Rebuilt function list: ['add', 'multiply']


In [11]:
for p in ["Add 2 to 20.", "Multiply 6 by 7", "What is 12 times 12?"]:
    await plan_and_execute(p)
    print()

PROMPT: 'Add 2 to 20.'
  available tools: ['add', 'multiply']
  planned tool_call: add({'a': 2, 'b': 20})
  ✅ executed via MCP -> 22



PROMPT: 'Multiply 6 by 7'
  available tools: ['add', 'multiply']
  planned tool_call: multiply({'a': 6, 'b': 7})
  ✅ executed via MCP -> 42



PROMPT: 'What is 12 times 12?'
  available tools: ['add', 'multiply']
  planned tool_call: multiply({'a': 12, 'b': 12})
  ✅ executed via MCP -> 144



In [12]:
print("Server log (stderr of the MCP server):")
print(open("server.log").read()[-1500:])

Server log (stderr of the MCP server):
[server] Processing request of type ListResourcesRequest
[server] Processing request of type ListToolsRequest
[server] Processing request of type ReadResourceRequest
[server] Processing request of type ListToolsRequest
[server] Processing request of type CallToolRequest
[server] add(2, 20)
[server] Processing request of type ListToolsRequest
[server] Processing request of type CallToolRequest
[server] add(15, 27)
[server] Processing request of type ListToolsRequest
[server] Processing request of type ListToolsRequest
[server] Processing request of type ListResourcesRequest
[server] Processing request of type ListToolsRequest
[server] Processing request of type ReadResourceRequest
[server] Processing request of type ListToolsRequest
[server] Processing request of type CallToolRequest
[server] add(2, 20)
[server] Processing request of type ListToolsRequest
[server] Processing request of type CallToolRequest
[server] multiply(6, 7)
[server] Processin

## Summary of the outputs
- **Ex. 2:** the client spawned `mcp run server.py` over STDIO and completed the `initialize` handshake.
- **Ex. 3:** discovered the resource `info://about` and the tool `add`, whose `inputSchema` properties are `a` and `b` (both integers, both required).
- **Ex. 4:** `convert_to_llm_tool` maps each MCP tool to an OpenAI-style function spec (name, description, JSON-Schema parameters).
- **Ex. 5:** for *"Add 2 to 20."* the planner proposed `add({'a': 2, 'b': 20})`, the client executed it through MCP, and the result is **22**. A prompt with no matching tool returns a plain answer and no tool call.
- **Optional:** after adding `multiply`, the rebuilt function list contains `add` and `multiply`, and *"Multiply 6 by 7"* → **42**, *"12 times 12"* → **144**, without changing any client code, because tools are discovered dynamically.

This saved run uses the **stub planner** (no `GITHUB_TOKEN`). With a token, the same cells send the function specs to a real LLM, which returns `tool_calls` that the same executor runs.